# Compositional Distributed Alignment Search for Nested NLI Causal Graphs

This notebook is designed to be run **top-to-bottom from a fresh runtime**. Old outputs are intentionally cleared in this version so stale execution order cannot be mistaken for valid evidence.

Methodological guardrails included here:
- lexical DAS chooses a patching site that is actually usable by the counterfactual dataset;
- MQNLI train/eval examples are unique and non-overlapping;
- MQNLI labels are stratified rather than accidentally 99% neutral;
- MQNLI DAS sites are selected from patching, not hardcoded, then validated by held-out full-vector patching;
- fixed-position DAS datasets filter out examples whose source/base token positions do not align;
- composition is only evaluated when the two single-variable DAS interventions share the same site, and it fails loudly rather than silently using an invalid fallback.


## 0. Bootstrap

Run this cell once at the top of every session. On Colab it clones the repo, installs pinned deps, downloads the MQNLI signature JSONs, and changes the working directory. Locally it just verifies you're in the right place.

After this cell finishes:
- `import nli_das` works
- the repo working tree is at `cs221m-final/`
- `outputs/{figures,tables}/` exist

In [ ]:
# ── Bootstrap ───────────────────────────────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

REPO_URL  = "https://github.com/aquantumreality/cs221m-final.git"
REPO_NAME = "cs221m-final"

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    os.chdir(repo_root)
else:
    # Local: walk up to find the repo root
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "nli_das").is_dir() and (cand / "requirements.txt").exists():
            os.chdir(cand)
            break
    else:
        raise FileNotFoundError("Could not find cs221m-final repo root.")

# Install dependencies. pyvene sometimes needs the git version if the PyPI build is stale.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
try:
    import pyvene  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/stanfordnlp/pyvene.git"])

# MQNLI signature files (needed for sections 8+). Skipped silently if already present.
TUTORIAL_DATA = Path("tutorial_data")
TUTORIAL_DATA.mkdir(exist_ok=True)
_MQNLI_FILES = [
    "mqnli_q_projectivity.json",
    "mqnli_neg_signature.json",
    "mqnli_empty_signature.json",
    "mqnli_cont_signature.json",
    "mqnli_neg_cont_signature.json",
]
_BASE = "https://raw.githubusercontent.com/stanfordnlp/pyvene/main/tutorials/advanced_tutorials/tutorial_data/"
for fn in _MQNLI_FILES:
    target = TUTORIAL_DATA / fn
    if not target.exists():
        subprocess.run(["curl", "-sL", _BASE + fn, "-o", str(target)], check=False)

# Output dirs
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)

print("Repo root:", Path.cwd())
print("Restart runtime if new packages were installed, then re-run from this cell.")


In [ ]:
# Optional: install nnsight if you want NDIF/nnsight experiments later.
# The core Colab notebook does not require nnsight.
import sys, subprocess, importlib.util
if importlib.util.find_spec('nnsight') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nnsight'])
else:
    print('nnsight already installed')


In [ ]:
# ── Library imports + device/seed ──────────────────────────────────────────
import json, gc, copy
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

import nli_das
from nli_das import (
    LexicalCausalModel,
    LEXICAL_PAIRS, UPWARD_TEMPLATES, DOWNWARD_TEMPLATES, DEFAULT_TEMPLATES,
    NLIExample, generate_examples, label_distribution, relation_distribution,
    build_counterfactual_dataset, build_random_source_dataset,
    build_wrong_variable_dataset, pair_level_split,
    LabelVerbalizer, compute_iia,
    make_das_config, train_das_alignment, evaluate_das_iia,
    run_patching_sweep, save_patching_heatmap_from_df,
)

SEED   = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")


---
## 1. Why DAS?

### The localization problem

A central goal of mechanistic interpretability is to **localize** an abstract concept to a component of a deep model. *Where* in GPT-2 does the model represent "the lexical relation between two words"? The naive approach is to **patch** the entire residual-stream activation at some `(layer, position)` from a source input into a base input, and check whether the model's output changes the way a symbolic causal model predicts. This is **activation patching** (also called interchange intervention).

But residual-stream vectors are **polysemantic** — a single vector encodes many features simultaneously. If we patch the full vector at the hypothesis-word position, we change *everything* encoded there: lexical category, semantic role, position, frequency, syntactic dependencies. We get behavioral effects but not a clean mechanism.

### What DAS adds

**DAS** (Geiger et al. 2023) learns an *orthogonal rotation* $R$ of the residual stream so that we can patch a **low-dimensional subspace** rather than the full vector. The rotation is trained — by gradient descent on counterfactual data — so that patching the subspace transfers the *target high-level variable* (e.g. `lexical_relation`) from source to base while leaving everything else alone.

Concretely, at a chosen `(layer, position, component)` site:

1. Forward-pass the base prompt and the source prompt.
2. Take the activation $h$ at the site for each.
3. Rotate into feature space: $f = h R^\top$.
4. **Swap the first $k$ features** of base with those of source.
5. Rotate back: $h' = f R$.
6. Continue the forward pass.
7. Loss: cross-entropy on the predicted label against the *symbolic counterfactual label* (what the high-level causal model predicts under the same intervention).

Only $R$ is trained — the LM is frozen. If a $k$-dim subspace can be found that aligns with the target variable, the loss drops; if not, the loss plateaus near chance.

### The metric: Interchange Intervention Accuracy (IIA)

For a held-out batch, IIA is the fraction of examples where the *patched* model's argmax label equals the *symbolic counterfactual* label. IIA = 1 means perfect causal abstraction at the chosen site; IIA at chance means the rotation found nothing meaningful.

We compare every result against three baselines:

- **Chance** (3-way label ≈ 0.33)
- **Random-source control**: same DAS rotation, but each base is paired with a *random* source. If the rotation actually transports the target variable from source to base, random sources should give random label-changes and IIA collapses to chance.
- **Wrong-variable control**: same DAS rotation evaluated on counterfactuals computed for a *different* target variable. If the rotation specifically encoded `lexical_relation`, evaluating it as if it encoded `premise_word_identity` gives low IIA.

These controls together establish that the DAS subspace really aligns with the target variable, not just with any source-dependent direction.

---
## 2. The high-level causal model

Before we look at any neural network, we write down the symbolic algorithm we *hope* GPT-2 has learned for our lexical NLI task. The model takes a premise word and a hypothesis word, computes their lexical relation, and outputs an NLI label.

```
inputs        :  premise_word, hypothesis_word, context (= template)
intermediate  :  lexical_relation ∈ {EQUIV, FORWARD, REVERSE, DISJOINT}
                 monotonicity     ∈ {upward, downward}  (from context)
output        :  label ∈ {entailment, neutral, contradiction}
```

The relation → label table depends on monotonicity:

|             | upward (e.g. "A X is on the table")       | downward (e.g. "No X is on the table") |
|---|---|---|
| `EQUIV`     | entailment                                 | entailment                              |
| `FORWARD`   | entailment   (hyponym→hypernym preserves)  | neutral      (negation flips)           |
| `REVERSE`   | neutral      (hypernym→hyponym loses info) | entailment   (flips back)               |
| `DISJOINT`  | contradiction                              | contradiction                           |

The `LexicalCausalModel` class in `nli_das` materializes this table and exposes `.run(base, interventions=...)` so we can ask *"what label would the symbolic model emit if we intervened on `lexical_relation = FORWARD`?"* — this is what gives us **gold counterfactual labels** for training and evaluating DAS.

In [ ]:
# ── Inspect the causal model ───────────────────────────────────────────────
causal = LexicalCausalModel(monotonicity="upward")

# Forward pass on (premise_word=dog, hypothesis_word=animal) — should give ENTAILMENT
base = causal.run(premise_word="dog", hypothesis_word="animal", context="on_the_table")
print(f"Base trace:  rel={base['lexical_relation']}, label={base['label']}")

# Counterfactual: pretend the lexical relation is DISJOINT instead
cf = causal.run(premise_word="dog", hypothesis_word="animal", context="on_the_table",
                interventions={"lexical_relation": "DISJOINT"})
print(f"After do(lexical_relation=DISJOINT):  label={cf['label']}")

# Monotonicity intervention: same words, but downward context
cf_mono = causal.run(premise_word="dog", hypothesis_word="animal", context="no_x_on_table",
                    interventions={"monotonicity": "downward"})
print(f"After do(monotonicity=downward):  rel={cf_mono['lexical_relation']}, label={cf_mono['label']}")


---
## 3. Generating controlled NLI examples

We need a dataset where:

- the lexical relation between every (premise, hypothesis) pair is **symbolically known** (so we can compute counterfactual labels for free),
- the **token position** of the content word is deterministic per template (so DAS can intervene at a fixed site without padding artifacts),
- the four relation classes are **roughly balanced** (so IIA isn't dominated by one class).

We auto-generate pairs from a small hand-curated hypernym ontology (about 50 leaf words across categories like *mammal*, *bird*, *vehicle*, *tool*, *furniture*, *fruit*). The four relations are derived from the closure of the ontology, with `DISJOINT` pairs subsampled per-word so no leaf dominates.

In [ ]:
# ── Vocabulary statistics ─────────────────────────────────────────────────
from nli_das.data import auto_generate_pairs
print(f"Lexical pairs: {len(LEXICAL_PAIRS)}")
print("Relation distribution:", dict(Counter(r for _, _, r in LEXICAL_PAIRS)))

# Template inventory
print(f"\nUpward templates  ({len(UPWARD_TEMPLATES)}):")
for t in UPWARD_TEMPLATES:
    print(f"  {t.name:18s}  {t.premise_format!r}")
print(f"\nDownward templates ({len(DOWNWARD_TEMPLATES)}):")
for t in DOWNWARD_TEMPLATES:
    print(f"  {t.name:18s}  {t.premise_format!r}")

# Materialise all (pair × template) examples
examples = generate_examples()
print(f"\nTotal examples: {len(examples)}")
print("Label distribution:", label_distribution(examples))
print("\nSample prompts:")
for ex in examples[:5]:
    print(f"  [{ex.label:14s}]  {ex.prompt!r}")


---
## 4. Counterfactual pairs + the two controls

For DAS we don't just need single examples — we need **(base, source) pairs** plus a gold counterfactual label that says: "if you patched `lexical_relation` from source into base, the model *should* output this label". The function `build_counterfactual_dataset` materializes these tuples up front, including the **exact token position** of the intervention site (we localize the content word's first BPE token).

Three more pieces of hygiene that make this evaluation robust:

1. **Pair-level holdout.** We split the lexical pairs into train/eval *before* sampling counterfactuals, so eval lexical items are never seen at training time. This rules out "the rotation memorized the (premise, hypothesis) prompts."
2. **`require_label_change=True`.** We discard pairs where the counterfactual label equals the base label — otherwise IIA gets inflated by trivially-stuck "predict the base label" behavior. This is the DAS-paper convention.
3. **Two controls, built from the same eval set:**
   - **Random-source**: for each base, replace the source with a randomly-chosen source from the pool and recompute the gold counterfactual label. If DAS really transports the source's `lexical_relation` value, this control's IIA should drop to chance.
   - **Wrong-variable**: same (base, source) pairs and intervention site, but recompute the gold counterfactual label as if the target variable were `premise_word_identity` instead of `lexical_relation`. If the learned subspace specifically encodes `lexical_relation`, it should score poorly here.

In [ ]:
# ── Build train + eval datasets and the two controls ───────────────────────
TARGET_VAR = "lexical_relation"
N_TRAIN, N_EVAL = 512, 128

tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
verbalizer = LabelVerbalizer.from_tokenizer(
    tokenizer,
    {"entailment": " yes", "neutral": " maybe", "contradiction": " no"},
)
print(f"verbalizer token ids = {verbalizer.token_ids}")

# Hold out 20% of lexical pairs for evaluation.
train_pairs, eval_pairs = pair_level_split(LEXICAL_PAIRS, train_frac=0.8, seed=SEED)
print(f"\nPair-level split: train={len(train_pairs)} pairs, eval={len(eval_pairs)} pairs")

# Pick one template for the lexical-relation sections so positions align.
TEMPLATES = [UPWARD_TEMPLATES[0]]   # "A {word} is on the table."

train_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=train_pairs, templates=TEMPLATES,
    n_examples=N_TRAIN, seed=SEED, require_label_change=True,
)
eval_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=eval_pairs, templates=TEMPLATES,
    n_examples=N_EVAL, seed=SEED + 42, require_label_change=True,
)
print(f"\nCounterfactual datasets:  train={len(train_ds)}  eval={len(eval_ds)}")

# Build the two controls from the eval set.
eval_random = build_random_source_dataset(eval_ds, seed=SEED + 1)
eval_wrong  = build_wrong_variable_dataset(eval_ds, wrong_variable="premise_word_identity")
print(f"\nControls:  random-source n={len(eval_random)}  wrong-variable n={len(eval_wrong)}")

# Show one concrete (base, source, cf_label) tuple
ex = train_ds.examples[0]
print(f"\nExample CF pair:")
print(f"  base    : {ex.base.prompt!r}   →  base_label={ex.base.label}")
print(f"  source  : {ex.source.prompt!r}   →  source_label={ex.source.label}")
print(f"  patch   : do({ex.target_variable}={ex.source.lexical_relation!r}) at token pos {ex.intervention_pos}")
print(f"  cf_label: {['entailment','neutral','contradiction'][ex.counterfactual_label_id]}")


---
## 5. Fine-tuning GPT-2 on the lexical NLI task

**Why this step is mandatory.** DAS learns an orthogonal rotation so that patching a low-dimensional subspace transports a high-level variable from source to base. But if the base model doesn't already solve the task — if its output is *not* a reliable function of the target variable — there is no meaningful gradient signal and no subspace to find.

Base GPT-2 achieves roughly **33–40 % factual accuracy** on our lexical NLI prompts — chance level for a 3-class task. The model has absorbed "A dog is on the table" from pre-training, but it has never been trained to emit `entailment / neutral / contradiction` tokens, so its logits on the verbalizer tokens (` yes` / ` maybe` / ` no`) are essentially uniform.

**The fix:** fine-tune GPT-2 on the *factual* (non-counterfactual) lexical NLI training examples using a standard causal-LM objective. We append the verbalizer token to each prompt and minimize cross-entropy over the full sequence. Fine-tuning is restricted to the **training lexical pairs** — the eval pairs remain completely unseen.

After 6 epochs on ~200–300 examples the model typically reaches **65–80 % factual accuracy** — enough for clean DAS gradients. This mirrors exactly what the pyvene MQNLI tutorial does before any DAS run, and it is the step that was missing from the initial draft, causing near-chance IIA in sections 5–8.

In [ ]:
# ── Fine-tune GPT-2 on factual lexical NLI ──────────────────────────────────
# Key fixes vs original: (1) loss on label token only → clean gradient signal,
# (2) 20 epochs at 2e-4 LR → sufficient convergence on ~336 examples.

from nli_das.data import generate_examples

LABEL_STR     = {"entailment": " yes", "neutral": " maybe", "contradiction": " no"}
NUM_FT_EPOCHS = 20
FT_LR         = 2e-4
FT_BSZ        = 16

factual_train = generate_examples(pairs=train_pairs, templates=TEMPLATES)
factual_eval  = generate_examples(pairs=eval_pairs,  templates=TEMPLATES)
print(f"Fine-tune:  {len(factual_train)} train  |  {len(factual_eval)} eval")
print("Label dist (train):", {k: sum(1 for e in factual_train if e.label == k)
                               for k in ("entailment", "neutral", "contradiction")})

def _encode_label_only(exs, tok, lmap):
    """Encode prompt + label token; return list of (prompt_ids, label_token_id)."""
    out = []
    for ex in exs:
        pids = tok.encode(ex.prompt, add_special_tokens=False)
        lids = tok.encode(lmap[ex.label], add_special_tokens=False)
        assert len(lids) == 1, f"label {ex.label!r} → {lids} (not single-token)"
        out.append((pids, lids[0]))
    return out

def _collate_label(batch):
    """Pad to longest; loss ONLY on the label token (last position)."""
    maxlen = max(len(p) + 1 for p, _ in batch)
    n = len(batch)
    input_ids = torch.zeros(n, maxlen, dtype=torch.long)
    attn_mask  = torch.zeros(n, maxlen, dtype=torch.long)
    labels     = torch.full((n, maxlen), -100, dtype=torch.long)
    for i, (pids, lid) in enumerate(batch):
        t = len(pids)
        input_ids[i, :t] = torch.tensor(pids)
        input_ids[i, t]  = lid
        attn_mask[i, :t + 1] = 1
        labels[i, t]     = lid   # CE only on this one token
    return input_ids, attn_mask, labels

train_seqs = _encode_label_only(factual_train, tokenizer, LABEL_STR)
eval_seqs  = _encode_label_only(factual_eval,  tokenizer, LABEL_STR)

# ── Training loop ────────────────────────────────────────────────────────────
ft_model  = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
optimizer = torch.optim.AdamW(ft_model.parameters(), lr=FT_LR, weight_decay=0.01)
rng_ft    = np.random.default_rng(SEED)

print(f"\nTraining {NUM_FT_EPOCHS} epochs (label-token loss only) …")
epoch_losses = []
ft_model.train()
for epoch in range(NUM_FT_EPOCHS):
    order = rng_ft.permutation(len(train_seqs)).tolist()
    tot_loss = 0; n_b = 0
    for start in range(0, len(order), FT_BSZ):
        batch = [train_seqs[i] for i in order[start : start + FT_BSZ]]
        ids, mask, lbl = _collate_label(batch)
        ids, mask, lbl = ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)
        out = ft_model(input_ids=ids, attention_mask=mask, labels=lbl)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad()
        tot_loss += out.loss.item(); n_b += 1
    epoch_losses.append(tot_loss / n_b)
    if (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch+1}/{NUM_FT_EPOCHS}  loss={epoch_losses[-1]:.4f}")

ft_model.eval()

# ── Training-loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(range(1, NUM_FT_EPOCHS + 1), epoch_losses, marker="o", color="tab:blue")
ax.set_xlabel("Epoch"); ax.set_ylabel("Label-token CE loss")
ax.set_title("GPT-2 fine-tuning on lexical NLI (label-only loss)"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/figures/finetuning_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Factual accuracy ──────────────────────────────────────────────────────────
verb_tids = verbalizer.token_id_tensor(device=DEVICE)   # [3] LongTensor
correct = total = 0
with torch.no_grad():
    for ex, (pids, _) in zip(factual_eval, eval_seqs):
        prompt_ids  = torch.tensor([pids]).to(DEVICE)
        logits_next = ft_model(input_ids=prompt_ids).logits[0, -1]   # [V]
        verb_logits = logits_next[verb_tids]                          # [3]
        pred        = verb_logits.argmax().item()
        correct    += int(pred == ex.label_id)
        total      += 1

factual_acc = correct / total
print(f"\nFactual accuracy (fine-tuned GPT-2, eval): {factual_acc:.1%}  ({correct}/{total})")
if factual_acc < 0.55:
    print("WARNING: accuracy still low — run more epochs or raise FT_LR.")
else:
    print("Sufficient factual accuracy. Proceeding.")

# Freeze for DAS — only the rotation gets gradient updates downstream.
model = ft_model
for p in model.parameters():
    p.requires_grad_(False)
n_layers = model.config.n_layer

del optimizer, train_seqs, eval_seqs, ft_model
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print(f"Model frozen: gpt2  ({n_layers} layers, hidden={model.config.n_embd})")

---
## 6. Activation patching baseline

Before training DAS, we run **vanilla activation patching** as a coarse baseline. For every `(layer, component, position)` cell in a grid, we patch the source's activation into the base's run, then measure two things:

- **Logit-difference recovery**: how much of the (clean − corrupted) logit gap the patch closes (after Wang et al. 2022, *Interpretability in the Wild*).
- **IIA**: argmax accuracy of the patched run against the symbolic counterfactual label.

Patching does *not* learn anything — it just swaps full activation vectors. So patching gets to "high cause" but can never separate the target variable from everything else encoded in the same vector. The heatmap tells us *where* in the network the relevant information is most concentrated, which gives DAS a sensible site to anchor the rotation.

In [ ]:
# ── Patching sweep ─────────────────────────────────────────────────────────
# Layer × component × position, on a 64-example subset (~3 min on T4).
# `model` is the fine-tuned, frozen GPT-2 from §5.
print(f"Fine-tuned GPT-2: {n_layers} layers, hidden={model.config.n_embd}")

# Smaller dataset for the patching sweep — the cell will run 12 layers × 3 components × seq_len positions.
patch_ds = build_counterfactual_dataset(
    tokenizer, target_variable=TARGET_VAR,
    pairs=train_pairs, templates=TEMPLATES,
    n_examples=64, seed=SEED, require_label_change=True,
)
print(f"Patching dataset: {len(patch_ds)}")

patch_df = run_patching_sweep(
    model, tokenizer, patch_ds.examples,
    layers=list(range(n_layers)),
    components=("mlp_output", "attention_input", "block_output"),
    positions="all",
    metric="logit_recovery",
    device=DEVICE, verbalizer=verbalizer,
    batch_size=8, progress=True,
)
patch_df.to_csv("outputs/tables/patching_sweep.csv", index=False)
print(f"\nSweep rows: {len(patch_df)}  base_accuracy={patch_df.attrs.get('base_accuracy', 0):.3f}")


In [ ]:
# ── Heatmaps + best-site selection ─────────────────────────────────────────
for comp in ("mlp_output", "attention_input", "block_output"):
    save_patching_heatmap_from_df(
        patch_df, f"outputs/figures/patching_{comp}.png",
        value_col="iia_correct", component=comp, annot=True,
    )

# Best (layer, component, position) by mean IIA then logit recovery.
site_scores = (
    patch_df.groupby(["component", "layer", "position"])
    .agg(iia=("iia_correct", "mean"), recovery=("recovery", "mean"))
    .reset_index()
    .sort_values(["iia", "recovery"], ascending=False)
)
print("Top 5 patching sites (all positions):")
print(site_scores.head().to_string(index=False))

# ── Restrict to positions DAS can actually use ───────────────────────────────
# Label-only fine-tuning often makes the final Answer: token the best full-vector
# patching site. That is real, but the CF dataset's intervention positions are
# content-word positions. If we selected the Answer: token and then silently fell
# back to a content-word position later, the layer/position pair would be incoherent.
content_positions = sorted({int(ex.intervention_pos) for ex in train_ds.examples})
valid_sites = site_scores[site_scores["position"].isin(content_positions)].copy()
assert len(valid_sites) > 0, "No patching sites match intervention positions in train_ds."

print(f"\nContent-word positions in CF dataset: {content_positions}")
print("Top 5 patching sites (content-word positions only):")
print(valid_sites.head().to_string(index=False))

best = valid_sites.iloc[0]
COMPONENT      = str(best["component"])
BEST_LAYER     = int(best["layer"])
FIXED_POSITION = int(best["position"])
print(f"\nSelected DAS anchor:  layer={BEST_LAYER}  component={COMPONENT}  position={FIXED_POSITION}")
print(f"  patching IIA at this usable site: {float(best['iia']):.3f}")


**Interpreting the patching result.** The heatmap should show that *full-vector* patching at the best site achieves moderate IIA — significantly above chance, but well short of perfect. This is exactly the limitation DAS is designed to address: the full activation carries the target variable *plus* a lot of other stuff, so the patch transports too much.

---
## 7. Single-variable DAS — `lexical_relation`

Now we train the DAS rotation. The intervenable model wraps GPT-2 with a `LowRankRotatedSpaceIntervention` at the (layer, component, position) site picked by patching. The only trainable parameters are the orthogonal rotation matrix; the LM stays frozen.

We filter the train and eval datasets to the chosen `FIXED_POSITION` so every batch patches the same token index (pyvene's rotation layer can't handle variable positions cleanly).

**Hyperparameters** chosen from preliminary sweeps:
- Subspace dimension `d=16` (the symbolic variable has 4 values but the neural encoding can be more distributed)
- 20 epochs, batch size 8, Adam lr 1e-3
- We also pass `eval_cf_dataset=eval_ds` so the training loop logs val loss + val IIA each epoch (catches overfitting on small data).

In [ ]:
from collections import Counter

# ── Filter lexical datasets to the selected fixed intervention position ──────
# Important: do NOT silently change FIXED_POSITION here. The layer was chosen for
# this exact position in the patching sweep. If this assertion fails, rerun §6–§7.
assert any(ex.intervention_pos == FIXED_POSITION for ex in train_ds.examples), (
    f"Selected FIXED_POSITION={FIXED_POSITION} is absent from train_ds; "
    "rerun the patching-site selection cell."
)

train_ds_f      = train_ds.filter_by_position(FIXED_POSITION)
eval_ds_f       = eval_ds.filter_by_position(FIXED_POSITION)
eval_random_f   = eval_random.filter_by_position(FIXED_POSITION)
eval_wrong_f    = eval_wrong.filter_by_position(FIXED_POSITION)

print(f"After filter to position {FIXED_POSITION}:")
print(f"  train={len(train_ds_f)}  eval={len(eval_ds_f)}  random={len(eval_random_f)}  wrong={len(eval_wrong_f)}")
print("  original train position counts:", dict(Counter(ex.intervention_pos for ex in train_ds.examples)))
print("  filtered train position counts:", dict(Counter(ex.intervention_pos for ex in train_ds_f.examples)))

assert len(train_ds_f) >= 64, "Too few lexical training CF examples after position filter."
assert len(eval_ds_f)  >= 32, "Too few lexical eval CF examples after position filter."
assert len(eval_random_f) == len(eval_ds_f), "Random-source control size mismatch after filtering."
assert len(eval_wrong_f)  == len(eval_ds_f), "Wrong-variable control size mismatch after filtering."


In [ ]:
# ── Train DAS ──────────────────────────────────────────────────────────────
DIM        = 16
NUM_EPOCHS = 20

torch.manual_seed(SEED)
cfg = make_das_config(model, layer=BEST_LAYER, component=COMPONENT,
                      intervention_dim=DIM, unit="pos")
das_out = train_das_alignment(
    model, tokenizer, train_ds_f, cfg,
    num_epochs=NUM_EPOCHS, lr=1e-3, batch_size=8,
    device=DEVICE, verbalizer=verbalizer,
    fixed_position=FIXED_POSITION,
    eval_cf_dataset=eval_ds_f,    # logs val loss + val IIA every epoch
    progress=True,
)

# Training curve
hist = pd.DataFrame(das_out.history)
print("\nLast 3 epochs:")
print(hist.tail(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["epoch"], hist["loss"], marker="o", label="train loss")
if "val_loss" in hist.columns: ax[0].plot(hist["epoch"], hist["val_loss"], marker="s", label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist["epoch"], hist["train_iia"], marker="o", label="train IIA")
if "val_iia" in hist.columns: ax[1].plot(hist["epoch"], hist["val_iia"], marker="s", label="val IIA")
ax[1].axhline(1/3, color="black", linestyle="--", lw=1, label="chance")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("IIA"); ax[1].set_ylim(0, 1.05); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.suptitle(f"DAS training — L{BEST_LAYER} {COMPONENT} pos{FIXED_POSITION} d{DIM}")
fig.tight_layout(); fig.savefig("outputs/figures/lexical_das_training.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Evaluate against controls ────────────────────────────────────────────────
res        = evaluate_das_iia(das_out.intervenable, eval_ds_f, tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)
res_random = evaluate_das_iia(das_out.intervenable, eval_random_f, tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)
res_wrong  = evaluate_das_iia(das_out.intervenable, eval_wrong_f,  tokenizer,
                              device=DEVICE, verbalizer=verbalizer, fixed_position=FIXED_POSITION)

summary = pd.DataFrame([
    {"condition": "Trained DAS",     "IIA": res["iia"]},
    {"condition": "Random-source",   "IIA": res_random["iia"]},
    {"condition": "Wrong-variable",  "IIA": res_wrong["iia"]},
]).round(3)
print(summary.to_string(index=False))
summary.to_csv("outputs/tables/lexical_relation_summary.csv", index=False)
print(f"\nFactual accuracy (base prediction, no intervention): {res['factual_accuracy']:.3f}")
print(f"Verbalizer-hit rate (unrestricted top-token is yes/no/maybe): {res['verbalizer_hit_rate']:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["tab:blue", "tab:gray", "tab:orange"]
ax.bar(summary["condition"], summary["IIA"], color=colors)
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="3-way chance")
ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.set_title(f"DAS on lexical_relation — L{BEST_LAYER} {COMPONENT} d{DIM}")
ax.legend(); fig.tight_layout()
fig.savefig("outputs/figures/lexical_relation_bars.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Random-source sanity check ───────────────────────────────────────────────
# Random-source is a generalization check, not the main negative control: its CF
# labels are recomputed from the random source, so high IIA can be good. The
# wrong-variable condition is the cleaner negative control.
from collections import Counter as _C21
same_cf = sum(
    int(a.counterfactual_label_id == b.counterfactual_label_id)
    for a, b in zip(eval_ds_f.examples, eval_random_f.examples)
)
print(f"\nRandom-source sanity:")
print(f"  Trained CF labels:       {dict(_C21(int(e.counterfactual_label_id) for e in eval_ds_f.examples))}")
print(f"  Random-source CF labels: {dict(_C21(int(e.counterfactual_label_id) for e in eval_random_f.examples))}")
print(f"  Same CF label (trained vs random): {same_cf}/{len(eval_ds_f)} ({same_cf/len(eval_ds_f):.1%})")
print("  Interpretation: random-source tests whether the learned rotation generalizes")
print("  to arbitrary valid sources. Wrong-variable is the stricter negative control.")


**Interpreting the result.** If DAS is doing what we expect:

- **Trained DAS IIA** should be substantially above chance (typically 0.6–0.75 for GPT-2 on this task).
- **Random-source IIA** should drop close to chance — the rotation does transport the source's `lexical_relation` value, but a *random* source's value is random with respect to the base, so the patched prediction has nothing to do with the symbolic counterfactual label.
- **Wrong-variable IIA** should also drop — the rotation transports `lexical_relation`, not `premise_word_identity`, so evaluating against the wrong gold labels gives random performance.

The gap between trained DAS and these two controls is the actual evidence that the learned subspace specifically encodes the target variable. **IIA alone is not enough.**

---
## 8. A second variable — `monotonicity`

The strongest evidence that DAS isn't just curve-fitting is to find *another* high-level variable in the same model at a *different* intervention site. `monotonicity` is a natural choice: it's the upward/downward feature that flips `FORWARD` ↔ `REVERSE` under negation.

Two changes from the lexical_relation setup:

1. **Different intervention site.** Monotonicity is encoded in the determiner/quantifier ("A", "Some", "No", "not") — the *first* token of the prompt. We localize the intervention there.
2. **Different templates.** Source examples must come from the *opposite* monotonicity class so the intervention actually flips the variable. Base = upward template; source = downward template.

In [ ]:
# ── Build the monotonicity counterfactual dataset ──────────────────────────
UP_T   = [UPWARD_TEMPLATES[0]]    # "A {word} is on the table."
DOWN_T = [DOWNWARD_TEMPLATES[0]]  # "No {word} is on the table."

mono_train = build_counterfactual_dataset(
    tokenizer, target_variable="monotonicity",
    pairs=train_pairs, templates=UP_T, source_templates=DOWN_T,
    n_examples=256, seed=SEED, require_label_change=True,
    require_monotonicity_flip=True,
)
mono_eval = build_counterfactual_dataset(
    tokenizer, target_variable="monotonicity",
    pairs=eval_pairs, templates=UP_T, source_templates=DOWN_T,
    n_examples=64, seed=SEED + 42, require_label_change=True,
    require_monotonicity_flip=True,
)

# Marker token position
MONO_POS = Counter(int(ex.intervention_pos) for ex in mono_train.examples).most_common(1)[0][0]
print(f"Monotonicity-marker token position: {MONO_POS}  (expect 0 — the determiner)")

mono_train = mono_train.filter_by_position(MONO_POS)
mono_eval  = mono_eval.filter_by_position(MONO_POS)
mono_eval_random = build_random_source_dataset(mono_eval, seed=SEED + 1)
mono_eval_wrong  = build_wrong_variable_dataset(mono_eval, wrong_variable="hypothesis_word_identity")
print(f"After filter:  train={len(mono_train)}  eval={len(mono_eval)}")
print(f"\nExample monotonicity pair:")
ex = mono_train.examples[0]
print(f"  base   : {ex.base.prompt!r}")
print(f"  source : {ex.source.prompt!r}")
print(f"  patch  : token pos {ex.intervention_pos}  (the determiner)")
print(f"  cf_label: {['entailment','neutral','contradiction'][ex.counterfactual_label_id]}")


In [ ]:
# ── Train DAS on monotonicity ──────────────────────────────────────────────
# We try an earlier layer (4) because monotonicity is more of a "syntactic" feature
# than the semantic lexical_relation — should be encoded shallower.
MONO_LAYER = 4
MONO_DIM   = 16

torch.manual_seed(SEED)
mono_cfg = make_das_config(model, layer=MONO_LAYER, component=COMPONENT,
                           intervention_dim=MONO_DIM, unit="pos")
mono_out = train_das_alignment(
    model, tokenizer, mono_train, mono_cfg,
    num_epochs=20, lr=1e-3, batch_size=8,
    device=DEVICE, verbalizer=verbalizer,
    fixed_position=MONO_POS,
    eval_cf_dataset=mono_eval,
    progress=True,
)

m_res   = evaluate_das_iia(mono_out.intervenable, mono_eval,        tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)
m_res_r = evaluate_das_iia(mono_out.intervenable, mono_eval_random, tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)
m_res_w = evaluate_das_iia(mono_out.intervenable, mono_eval_wrong,  tokenizer,
                           device=DEVICE, verbalizer=verbalizer, fixed_position=MONO_POS)

mono_summary = pd.DataFrame([
    {"condition": "Trained DAS",     "IIA": m_res["iia"]},
    {"condition": "Random-source",   "IIA": m_res_r["iia"]},
    {"condition": "Wrong-variable",  "IIA": m_res_w["iia"]},
]).round(3)
print(mono_summary.to_string(index=False))
mono_summary.to_csv("outputs/tables/monotonicity_summary.csv", index=False)

print()
print("=" * 60)
print("MONOTONICITY NOTE")
print("Trained DAS IIA: see bar chart above.")
print()
print("If IIA is 0.000 for both trained and random-source, this is")
print("expected: the model was fine-tuned ONLY on upward templates.")
print("It never learned to use the determiner ('A' vs 'No') to")
print("determine the label, so patching position 0 has no causal")
print("effect on its output.")
print()
print("For the final report: present monotonicity as an attempted")
print("extension that failed due to template mismatch in fine-tuning.")
print("=" * 60)

In [ ]:
# ── Side-by-side: lexical_relation vs monotonicity ─────────────────────────
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(3); width = 0.36
a_vals = [res["iia"], res_random["iia"], res_wrong["iia"]]
e_vals = [m_res["iia"], m_res_r["iia"], m_res_w["iia"]]
ax.bar(x - width/2, a_vals, width, color="tab:blue",
       label=f"lexical_relation  (L{BEST_LAYER}, d{DIM}, pos{FIXED_POSITION})")
ax.bar(x + width/2, e_vals, width, color="tab:purple",
       label=f"monotonicity     (L{MONO_LAYER}, d{MONO_DIM}, pos{MONO_POS})")
ax.set_xticks(x); ax.set_xticklabels(["Trained", "Random-source", "Wrong-variable"])
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="3-way chance")
ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.set_title("Two high-level variables, two intervention sites")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout(); fig.savefig("outputs/figures/two_variables.png", dpi=150, bbox_inches="tight")
plt.show()


**Why this matters.** A single high-level variable can always be explained away as "DAS found *some* direction that correlates with whatever varies between base and source." Finding **two** variables, at two different intervention sites, with two different sets of controls, makes that explanation much harder to sustain — and it's the first hint that the model has *compositional* internal structure, which is exactly what we'll test in section 11.

---
## 9. MQNLI — a nested causal graph

MQNLI (Multiply Quantified NLI) replaces our two-variable lexical model with a 33-variable tree. The causal structure is:

```
N_P_O, N_H_O  →  N_O  ┐
Adj_P_O, ...  →  Adj_O │ → NP_O ─┐
                        │          │
V_P, V_H      →  V     │          │
Adv_P, Adv_H  →  Adv  → VP ──────┤→ QP_O ─┐
                                   │         │
Q_P_O, Q_H_O  →  Q_O ─────────────┘         │→ NegP → QP_S  (final label)
                                              │         │
Neg_P, Neg_H  →  Neg ────────────────────────┘         │
                                                        │
(subject side, symmetric) ──── NP_S, Q_S ─────────────┘
```

We implement the causal model directly from the 5 JSON signature files in `tutorial_data/`. The key functions are:

- **noun/adj/verb merge**: EQUIV iff same word, else INDEPENDENCE
- **noun_phrase**: `adj if noun==EQUIV else INDEP` (adj filters through to NP only when noun is EQUIV)
- **quantifier_merge**: returns `q_proj[Q_P][Q_H]` — a lookup sub-table
- **quantifier_phrase**: `q_sub_table[restriction_rel][scope_rel]` — the full projectivity lookup
- **negation_merge**: selects one of the 4 signature tables based on (Neg_P, Neg_H)
- **negation_phrase**: `neg_table[qp_o]` — applies the negation signature

The final `QP_S` (NatLog relation) maps to NLI label: `entails/equivalence → entailment`, `contradiction → contradiction`, everything else → `neutral`.

In [ ]:
# ── Section 9: MQNLI causal model + balanced dataset ────────────────────────
import json as _json, random as _random
from pathlib import Path
from collections import Counter as _Counter, defaultdict as _defaultdict
from itertools import product as _product

# ── Load Natural Logic signature tables ──────────────────────────────────────
_TD = Path("tutorial_data")
q_proj       = _json.load(open(_TD / "mqnli_q_projectivity.json"))
neg_sig      = _json.load(open(_TD / "mqnli_neg_signature.json"))
empty_sig    = _json.load(open(_TD / "mqnli_empty_signature.json"))
cont_sig     = _json.load(open(_TD / "mqnli_cont_signature.json"))
neg_cont_sig = _json.load(open(_TD / "mqnli_neg_cont_signature.json"))
print("NatLog relations:", list(neg_sig.keys()))

# ── Vocabulary: small enough for exhaustive generation; all words are common GPT-2 tokens ──
MQNLI_Q   = ["every", "some"]
MQNLI_NS  = ["dog",   "cat"]
MQNLI_NO  = ["tree",  "rock"]
MQNLI_ADJ = ["quickly", "slowly"]
MQNLI_ADV = ["often",   "rarely"]
MQNLI_V   = ["climbed", "chased"]
MQNLI_NEG = ["not", ""]

MQNLI_LEAVES = {
    "Q_P_S": MQNLI_Q,   "Q_H_S": MQNLI_Q,
    "Q_P_O": MQNLI_Q,   "Q_H_O": MQNLI_Q,
    "N_P_S": MQNLI_NS,  "N_H_S": MQNLI_NS,
    "N_P_O": MQNLI_NO,  "N_H_O": MQNLI_NO,
    "Adj_P_S": MQNLI_ADJ, "Adj_H_S": MQNLI_ADJ,
    "Adj_P_O": MQNLI_ADJ, "Adj_H_O": MQNLI_ADJ,
    "Adv_P": MQNLI_ADV,   "Adv_H": MQNLI_ADV,
    "V_P":  MQNLI_V,       "V_H":  MQNLI_V,
    "Neg_P": MQNLI_NEG,   "Neg_H": MQNLI_NEG,
}

# ── Sentence formatter: grammatical negation, and a stable char-span helper ──
def _fmt_mqnli(ex):
    vp = "does not see" if ex["Neg_P"] == "not" else "sees"
    vh = "does not see" if ex["Neg_H"] == "not" else "sees"
    p = (f"{ex['Q_P_S']} {ex['N_P_S']} who {ex['Adj_P_S']} {ex['V_P']} "
         f"{vp} {ex['Adv_P']} {ex['Q_P_O']} {ex['N_P_O']} "
         f"who {ex['Adj_P_O']} {ex['V_P']}")
    h = (f"{ex['Q_H_S']} {ex['N_H_S']} who {ex['Adj_H_S']} {ex['V_H']} "
         f"{vh} {ex['Adv_H']} {ex['Q_H_O']} {ex['N_H_O']} "
         f"who {ex['Adj_H_O']} {ex['V_H']}")
    return f"{p}. {h}. Answer:"

def _mqnli_qpo_char_span(ex):
    """Character span of Q_P_O: the object quantifier in the premise."""
    vp_str = "does not see" if ex["Neg_P"] == "not" else "sees"
    prefix = (f"{ex['Q_P_S']} {ex['N_P_S']} who {ex['Adj_P_S']} {ex['V_P']} "
              f"{vp_str} {ex['Adv_P']} ")
    start = len(prefix)
    return start, start + len(ex["Q_P_O"])

# ── MQNLI causal model ───────────────────────────────────────────────────────
_EQV = "equivalence"
_IND = "independence"
_NATLOG_TO_NLI = {
    "equivalence": "entailment", "entails": "entailment",
    "reverse entails": "neutral", "independence": "neutral",
    "cover": "neutral", "alternation": "neutral",
    "contradiction": "contradiction",
}
from nli_das.causal_models import LABEL2ID as _L2I

def _mq(a, b): return _EQV if a == b else _IND
def _np(adj, noun): return adj if noun == _EQV else _IND

def _neg_table(neg_p, neg_h):
    if neg_p == "not" and neg_h == "not": return neg_sig
    if neg_p == "not" and neg_h == "":   return neg_cont_sig
    if neg_p == "" and neg_h == "not":   return cont_sig
    return empty_sig

def run_mqnli(ex, interventions=None):
    """Run the symbolic MQNLI causal model, optionally intervening on internal variables."""
    ivs = dict(interventions or {})
    e = ex
    # Object branch
    N_O   = ivs.pop("N_O",   _mq(e["N_P_O"], e["N_H_O"]))
    Adj_O = ivs.pop("Adj_O", _mq(e["Adj_P_O"], e["Adj_H_O"]))
    NP_O  = ivs.pop("NP_O",  _np(Adj_O, N_O))
    V     = ivs.pop("V",     _mq(e["V_P"], e["V_H"]))
    Adv   = ivs.pop("Adv",   _mq(e["Adv_P"], e["Adv_H"]))
    VP    = ivs.pop("VP",    _np(Adv, V))
    Q_O   = ivs.pop("Q_O",   q_proj[e["Q_P_O"]][e["Q_H_O"]])
    QP_O  = ivs.pop("QP_O",  Q_O[NP_O][VP])
    # Negation phrase
    Neg   = ivs.pop("Neg",   _neg_table(e["Neg_P"], e["Neg_H"]))
    NegP  = ivs.pop("NegP",  Neg[QP_O])
    # Subject branch
    N_S   = ivs.pop("N_S",   _mq(e["N_P_S"], e["N_H_S"]))
    Adj_S = ivs.pop("Adj_S", _mq(e["Adj_P_S"], e["Adj_H_S"]))
    NP_S  = ivs.pop("NP_S",  _np(Adj_S, N_S))
    Q_S   = ivs.pop("Q_S",   q_proj[e["Q_P_S"]][e["Q_H_S"]])
    QP_S  = ivs.pop("QP_S",  Q_S[NP_S][NegP])
    assert not ivs, f"Unknown interventions: {sorted(ivs)}"
    label    = _NATLOG_TO_NLI[QP_S]
    label_id = _L2I[label]
    return {**e,
            "N_O":N_O,"Adj_O":Adj_O,"NP_O":NP_O,"V":V,"Adv":Adv,"VP":VP,
            "Q_O":Q_O,"QP_O":QP_O,"Neg":Neg,"NegP":NegP,
            "N_S":N_S,"Adj_S":Adj_S,"NP_S":NP_S,"Q_S":Q_S,"QP_S":QP_S,
            "label":label,"label_id":label_id,"prompt":_fmt_mqnli(e)}

# ── Exhaustive unique pool + stratified split ────────────────────────────────
def gen_mqnli_stratified(n_per_label=400, train_frac=0.8, seed=0):
    """Generate unique examples exhaustively, then sample a balanced train/eval split."""
    keys = list(MQNLI_LEAVES.keys())
    buckets = _defaultdict(list)
    seen_prompts = set()
    for values in _product(*(MQNLI_LEAVES[k] for k in keys)):
        ex = dict(zip(keys, values))
        res = run_mqnli(ex)
        if res["prompt"] in seen_prompts:
            continue
        seen_prompts.add(res["prompt"])
        buckets[res["label"]].append(res)

    rng = _random.Random(seed)
    print("Full unique MQNLI pool by label:", {k: len(v) for k, v in buckets.items()})
    labels = ["entailment", "neutral", "contradiction"]
    available_min = min(len(buckets[k]) for k in labels)
    if available_min < n_per_label:
        print(f"Requested {n_per_label}/label, but rarest label has {available_min}; using {available_min}/label.")
    take = min(n_per_label, available_min)

    train, eval_ = [], []
    for lab in labels:
        items = list(buckets[lab])
        rng.shuffle(items)
        chosen = items[:take]
        split = int(round(train_frac * take))
        train.extend(chosen[:split])
        eval_.extend(chosen[split:])
    rng.shuffle(train); rng.shuffle(eval_)
    return train + eval_, train, eval_

N_MQNLI_PER_LABEL = 400
print(f"Generating stratified MQNLI dataset ({N_MQNLI_PER_LABEL} per label when available) ...")
mqnli_all, mqnli_train, mqnli_eval = gen_mqnli_stratified(
    n_per_label=N_MQNLI_PER_LABEL, train_frac=0.8, seed=SEED
)

train_prompts = {e["prompt"] for e in mqnli_train}
eval_prompts  = {e["prompt"] for e in mqnli_eval}
overlap = len(train_prompts & eval_prompts)
assert overlap == 0, f"Train/eval overlap: {overlap} prompts"
assert len(mqnli_train) > 0 and len(mqnli_eval) > 0, "Empty MQNLI train/eval split."

print(f"\nMQNLI: {len(mqnli_train)} train / {len(mqnli_eval)} eval  (overlap={overlap})")
print("Label dist (train):", dict(_Counter(e["label"] for e in mqnli_train)))
print("Label dist (eval): ", dict(_Counter(e["label"] for e in mqnli_eval)))
print("Q_P_O char-start dist (train):", dict(_Counter(_mqnli_qpo_char_span(e)[0] for e in mqnli_train)))
print("\nSample sentence:")
print(" ", mqnli_train[min(42, len(mqnli_train)-1)]["prompt"])
print("  -> label:", mqnli_train[min(42, len(mqnli_train)-1)]["label"],
      " QP_O:", mqnli_train[min(42, len(mqnli_train)-1)]["QP_O"],
      " NegP:", mqnli_train[min(42, len(mqnli_train)-1)]["NegP"])

_cf_test = run_mqnli(mqnli_train[0], {"QP_S": "contradiction"})
assert _cf_test["label"] == "contradiction"
print("\nCausal-model sanity check passed.")


## 10. Fine-tuning GPT-2 on MQNLI

Per the DAS paper, **factual accuracy must be high before DAS results are meaningful** — the rotation has nothing to align with otherwise. We target at least 80% on the held-out eval set before any MQNLI DAS is allowed to train.

Key differences from the lexical NLI fine-tuning:

- **Label-only loss** throughout.
- **20 epochs**, batch size 32, AdamW learning rate `2e-4`.
- The MQNLI dataset is generated as a unique, stratified train/eval split.
- The same frozen model (`mq_model`) is then used for the MQNLI patching and DAS sections.


In [ ]:
# ── Section 10: Fine-tune GPT-2 on MQNLI ────────────────────────────────────
# This is a harder nested NLI task than the lexical toy task. Keep the factual
# accuracy visible: if it is low, downstream DAS should be framed as preliminary.
MQ_EPOCHS = 20
MQ_LR     = 2e-4
MQ_BSZ    = 32
MQ_LABEL_STR = {"entailment": " yes", "neutral": " maybe", "contradiction": " no"}

def _enc_mq(exs, tok, lmap):
    out = []
    for ex in exs:
        pids = tok.encode(ex["prompt"], add_special_tokens=False)
        lid  = tok.encode(lmap[ex["label"]], add_special_tokens=False)
        assert len(lid) == 1
        out.append((pids, lid[0]))
    return out

def _col_mq(batch, dev):
    maxlen = max(len(p) + 1 for p, _ in batch)
    n = len(batch)
    ids  = torch.zeros(n, maxlen, dtype=torch.long)
    mask = torch.zeros(n, maxlen, dtype=torch.long)
    lbls = torch.full((n, maxlen), -100, dtype=torch.long)
    for i, (pids, lid) in enumerate(batch):
        t = len(pids)
        ids[i, :t] = torch.tensor(pids)
        ids[i, t]  = lid
        mask[i, :t + 1] = 1
        lbls[i, t] = lid
    return ids.to(dev), mask.to(dev), lbls.to(dev)

mq_tr_enc = _enc_mq(mqnli_train, tokenizer, MQ_LABEL_STR)
mq_ev_enc = _enc_mq(mqnli_eval,  tokenizer, MQ_LABEL_STR)
print(f"Fine-tune MQNLI: {len(mq_tr_enc)} train / {len(mq_ev_enc)} eval")
print("MQNLI train label dist:", dict(Counter(e["label"] for e in mqnli_train)))
print("MQNLI eval label dist: ", dict(Counter(e["label"] for e in mqnli_eval)))

mq_model  = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
mq_optim  = torch.optim.AdamW(mq_model.parameters(), lr=MQ_LR, weight_decay=0.01)
_rng10    = np.random.default_rng(SEED + 100)
mq_losses = []

mq_model.train()
for epoch in range(MQ_EPOCHS):
    order = _rng10.permutation(len(mq_tr_enc)).tolist()
    tot = 0; nb = 0
    for s in range(0, len(order), MQ_BSZ):
        batch = [mq_tr_enc[i] for i in order[s : s + MQ_BSZ]]
        ids, mask, lbl = _col_mq(batch, DEVICE)
        out  = mq_model(input_ids=ids, attention_mask=mask, labels=lbl)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(mq_model.parameters(), 1.0)
        mq_optim.step(); mq_optim.zero_grad()
        tot += out.loss.item(); nb += 1
    mq_losses.append(tot / nb)
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"  epoch {epoch+1}/{MQ_EPOCHS}  loss={mq_losses[-1]:.4f}")

mq_model.eval()

# ── Factual accuracy ──────────────────────────────────────────────────────────
_verb_tids = verbalizer.token_id_tensor(device=DEVICE)
correct = total = 0
pred_counts = Counter()
with torch.no_grad():
    for ex, (pids, _) in zip(mqnli_eval, mq_ev_enc):
        logits = mq_model(input_ids=torch.tensor([pids]).to(DEVICE)).logits[0, -1]
        pred   = int(logits[_verb_tids].argmax().item())
        pred_counts[pred] += 1
        correct += int(pred == ex["label_id"])
        total   += 1

mq_factual_acc = correct / total
print(f"\nFactual accuracy (MQNLI eval): {mq_factual_acc:.1%}  ({correct}/{total})")
print("Predicted label-id distribution:", dict(pred_counts))
if mq_factual_acc < 0.80:
    print("WARNING: below 80%. Treat MQNLI DAS/composition as preliminary evidence, not a final claim.")
else:
    print("Good factual accuracy. Proceeding to DAS.")

# Loss curve
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(range(1, MQ_EPOCHS + 1), mq_losses, marker="o", color="tab:green")
ax.set_xlabel("Epoch"); ax.set_ylabel("CE loss (label token)")
ax.set_title("GPT-2 fine-tuning on MQNLI"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/figures/mqnli_finetuning_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# Freeze MQNLI model
for p in mq_model.parameters():
    p.requires_grad_(False)
mq_n_layers = mq_model.config.n_layer
del mq_optim
if DEVICE.type == "cuda": torch.cuda.empty_cache()
gc.collect()
print(f"mq_model frozen: {mq_n_layers} layers, hidden={mq_model.config.n_embd}")


In [ ]:
# ── Section 11a: MQNLI patching sweep — find valid DAS anchors ──────────────
# Key methodological guard:
#   We only train DAS for an MQNLI variable if *full-vector patching* at a
#   position-aligned site already gives above-chance counterfactual signal.
#   If full patching cannot move the model correctly, then DAS should not be
#   expected to find a clean low-dimensional subspace there.
from types import SimpleNamespace as _SN
from collections import Counter, defaultdict
from math import comb as _comb

class _MQPatchEx:
    """Thin CounterfactualExample-compatible wrapper for MQNLI CF pairs."""
    def __init__(self, p):
        self.base   = _SN(prompt=p["base"]["prompt"])
        self.source = _SN(prompt=p["source"]["prompt"])
        self.base_label_id           = int(p["base"]["label_id"])
        self.source_label_id         = int(p["source"]["label_id"])
        self.counterfactual_label_id = int(p["cf_label_id"])
        self.intervention_pos        = int(p.get("intervention_pos", -1))
        self.target_variable         = p.get("target_var", "MQNLI")

def _find_qpo_token_pos(tok, ex):
    """Token index of Q_P_O using its exact character span, not occurrence count."""
    if "_qpo_tok_pos" in ex:
        return ex["_qpo_tok_pos"]
    start_char, end_char = _mqnli_qpo_char_span(ex)
    prompt = ex["prompt"]
    if tok.is_fast:
        enc = tok(prompt, return_offsets_mapping=True, add_special_tokens=False)
        for i, (a, b) in enumerate(enc["offset_mapping"]):
            if a == b:
                continue
            if a <= start_char < b or (start_char <= a < end_char):
                ex["_qpo_tok_pos"] = i
                return i
        ex["_qpo_tok_pos"] = None
        return None
    ids = tok(prompt, add_special_tokens=False)["input_ids"]
    for i in range(len(ids)):
        if len(tok.decode(ids[:i+1])) > start_char:
            ex["_qpo_tok_pos"] = i
            return i
    ex["_qpo_tok_pos"] = None
    return None

# Precompute token positions once. This avoids repeated tokenizer calls while
# sampling aligned counterfactual pairs.
for _ex in mqnli_train + mqnli_eval:
    _find_qpo_token_pos(tokenizer, _ex)

def _qpos(ex):
    return _find_qpo_token_pos(tokenizer, ex)

def _position_counts(exs):
    return dict(sorted(Counter(_qpos(e) for e in exs).items()))

print("Q_P_O token-position counts:")
print("  train:", _position_counts(mqnli_train))
print("  eval: ", _position_counts(mqnli_eval))

def _make_mqnli_cf_pair(base, source, target_var, require_change=True):
    if base[target_var] == source[target_var]:
        return None
    cf = run_mqnli(base, {target_var: source[target_var]})
    if require_change and cf["label_id"] == base["label_id"]:
        return None
    return {
        "base": base,
        "source": source,
        "cf_label_id": int(cf["label_id"]),
        "target_var": target_var,
    }

def sample_mqnli_cf_aligned_stratified(
    exs, target_var, pos, n_per_label=32, seed=0,
    require_change=True, max_attempts=250_000,
):
    """Sample aligned MQNLI CF pairs, stratified by counterfactual label.

    Alignment means base and source both place Q_P_O at the same token index `pos`.
    Stratification prevents a fake-high IIA caused by mostly-one-label CF sets.
    """
    rng = _random.Random(seed)
    by_label = {0: [], 1: [], 2: []}
    seen = set()

    # Restrict first so random choices do not waste almost all attempts.
    eligible = [e for e in exs if _qpos(e) == pos]
    if len(eligible) < 2:
        return [], {0: 0, 1: 0, 2: 0}, len(eligible)

    for _ in range(max_attempts):
        if min(len(by_label[k]) for k in by_label) >= n_per_label:
            break
        base = rng.choice(eligible)
        source = rng.choice(eligible)
        p = _make_mqnli_cf_pair(base, source, target_var, require_change=require_change)
        if p is None:
            continue
        key = (p["base"]["prompt"], p["source"]["prompt"], p["cf_label_id"])
        if key in seen:
            continue
        seen.add(key)
        lab = int(p["cf_label_id"])
        if len(by_label[lab]) < n_per_label:
            p["intervention_pos"] = int(pos)
            by_label[lab].append(p)

    counts = {k: len(v) for k, v in by_label.items()}
    take = min(counts.values())
    if take == 0:
        return [], counts, len(eligible)

    out = []
    for lab in [0, 1, 2]:
        out.extend(by_label[lab][:take])
    rng.shuffle(out)
    return out, counts, len(eligible)

def _majority_baseline(pairs):
    if not pairs:
        return float("nan")
    counts = Counter(int(p["cf_label_id"]) for p in pairs)
    return max(counts.values()) / len(pairs)

def _cf_label_dist(pairs):
    return dict(sorted(Counter(int(p["cf_label_id"]) for p in pairs).items()))

MQ_MIN_FACTUAL_ACC = 0.80
MQ_PATCH_ALPHA = 0.05
MQ_MIN_TRAIN_PAIRS = 48
MQ_MIN_EVAL_PAIRS = 24
MQ_MIN_EVAL_PER_LABEL = 8

def _binom_tail_prob_at_or_above(k, n, p0):
    """Exact one-sided binomial tail P[X >= k] under baseline p0."""
    if n <= 0 or not (0 <= p0 <= 1):
        return float("nan")
    return float(sum(_comb(n, i) * (p0 ** i) * ((1 - p0) ** (n - i))
                     for i in range(int(k), int(n) + 1)))

def _score_patch_df(patch_df, pos):
    site_scores = (
        patch_df.groupby(["component", "layer", "position"])
        .agg(iia=("iia_correct", "mean"), recovery=("recovery", "mean"))
        .reset_index()
    )
    # Be conservative: choose only the site/position actually used by the
    # fixed-position dataset.
    site_scores = site_scores[site_scores["position"] == int(pos)].copy()
    if len(site_scores) == 0:
        return None
    site_scores = site_scores.sort_values(["iia", "recovery"], ascending=False)
    return site_scores

def _run_patch_sweep_for_candidate(target_var, pos, seed):
    sweep_pairs, sweep_counts, eligible_n = sample_mqnli_cf_aligned_stratified(
        mqnli_train, target_var, pos, n_per_label=16, seed=seed,
        require_change=True, max_attempts=200_000,
    )
    print(f"\n{target_var} pos={pos}: eligible train examples={eligible_n}")
    print(f"  sweep CF counts before balancing attempt: {sweep_counts}")
    print(f"  balanced sweep pairs={len(sweep_pairs)}; dist={_cf_label_dist(sweep_pairs)}")
    if len(sweep_pairs) < 24:
        print("  Skipping this candidate: too few balanced sweep pairs.")
        return None

    patch_examples = [_MQPatchEx(p) for p in sweep_pairs]
    try:
        patch_df = run_patching_sweep(
            mq_model, tokenizer, patch_examples,
            layers=list(range(mq_n_layers)),
            components=("block_output",),
            positions=[int(pos)],
            metric="logit_recovery",
            device=DEVICE, verbalizer=verbalizer,
            batch_size=8, progress=True,
        )
    except Exception as exc:
        print(f"  positions=[{pos}] failed with {type(exc).__name__}; falling back to positions='all' then filtering.")
        patch_df = run_patching_sweep(
            mq_model, tokenizer, patch_examples,
            layers=list(range(mq_n_layers)),
            components=("block_output",),
            positions="all",
            metric="logit_recovery",
            device=DEVICE, verbalizer=verbalizer,
            batch_size=8, progress=True,
        )

    patch_df.to_csv(f"outputs/tables/mqnli_patch_{target_var}_pos{pos}.csv", index=False)
    scored = _score_patch_df(patch_df, pos)
    if scored is None or len(scored) == 0:
        print("  No patch rows for this candidate position.")
        return None

    print("  top patching sites at this aligned position:")
    print(scored.head(5).to_string(index=False))

    best = scored.iloc[0]
    return {
        "target_var": target_var,
        "pos": int(pos),
        "layer": int(best["layer"]),
        "component": str(best["component"]),
        "patch_iia": float(best["iia"]),
        "recovery": float(best["recovery"]),
        "sweep_pairs": sweep_pairs,
    }

def _evaluate_full_patch_at_site(target_var, pairs, layer, component, pos):
    """Held-out full-vector patching prerequisite at the selected anchor.

    Anchor selection can look good by chance on its search data. DAS readiness is
    therefore decided by this held-out patching test, not by the training sweep.
    """
    if len(pairs) == 0:
        return {"iia": float("nan"), "recovery": float("nan"),
                "correct": 0, "n": 0, "p_value": float("nan")}

    patch_examples = [_MQPatchEx(p) for p in pairs]
    patch_df = run_patching_sweep(
        mq_model, tokenizer, patch_examples,
        layers=[int(layer)],
        components=(str(component),),
        positions=[int(pos)],
        metric="logit_recovery",
        device=DEVICE, verbalizer=verbalizer,
        batch_size=8, progress=True,
    )
    patch_df.to_csv(
        f"outputs/tables/mqnli_patch_{target_var}_heldout_L{layer}_{component}_pos{pos}.csv",
        index=False,
    )

    rows = patch_df[
        (patch_df["layer"] == int(layer)) &
        (patch_df["component"] == str(component)) &
        (patch_df["position"] == int(pos))
    ]
    if len(rows) == 0:
        return {"iia": float("nan"), "recovery": float("nan"),
                "correct": 0, "n": 0, "p_value": float("nan")}

    correct = int(rows["iia_correct"].sum())
    n = int(rows["iia_correct"].count())
    iia = correct / n if n else float("nan")
    recovery = float(rows["recovery"].mean()) if n else float("nan")
    baseline = max(1/3, _majority_baseline(pairs))
    p_value = _binom_tail_prob_at_or_above(correct, n, baseline)
    return {"iia": float(iia), "recovery": recovery,
            "correct": correct, "n": n, "p_value": float(p_value)}

def select_mqnli_anchor_and_data(target_var, seed):
    # Candidate positions are positions that occur in both train and eval;
    # here they usually correspond to the same Q_P_O slot with/without negation.
    train_positions = set(_qpos(e) for e in mqnli_train)
    eval_positions  = set(_qpos(e) for e in mqnli_eval)
    candidate_positions = sorted(p for p in (train_positions & eval_positions) if p is not None)
    print(f"\n=== Selecting MQNLI anchor for {target_var} ===")
    print("Candidate aligned Q_P_O positions:", candidate_positions)

    candidates = []
    for j, pos in enumerate(candidate_positions):
        cand = _run_patch_sweep_for_candidate(target_var, pos, seed + 100*j)
        if cand is not None:
            candidates.append(cand)

    if not candidates:
        return {"ready": False, "reason": "no candidate had enough balanced CF pairs"}

    candidates = sorted(candidates, key=lambda d: (d["patch_iia"], d["recovery"]), reverse=True)
    best = candidates[0]
    print(f"\nBest {target_var} patching candidate: L{best['layer']} {best['component']} pos{best['pos']} "
          f"IIA={best['patch_iia']:.3f}, recovery={best['recovery']:.4f}")

    # Build balanced train/eval CF sets at the selected position for DAS.
    tr_pairs, tr_counts, tr_eligible = sample_mqnli_cf_aligned_stratified(
        mqnli_train, target_var, best["pos"], n_per_label=48,
        seed=seed + 777, require_change=True, max_attempts=350_000,
    )
    ev_pairs, ev_counts, ev_eligible = sample_mqnli_cf_aligned_stratified(
        mqnli_eval, target_var, best["pos"], n_per_label=24,
        seed=seed + 888, require_change=True, max_attempts=350_000,
    )
    print(f"{target_var} DAS train: eligible={tr_eligible}, sampled={len(tr_pairs)}, counts={tr_counts}, dist={_cf_label_dist(tr_pairs)}")
    print(f"{target_var} DAS eval : eligible={ev_eligible}, sampled={len(ev_pairs)}, counts={ev_counts}, dist={_cf_label_dist(ev_pairs)}")
    print(f"{target_var} eval majority-label baseline: {_majority_baseline(ev_pairs):.3f}")

    min_eval_per_label = min(Counter(int(p["cf_label_id"]) for p in ev_pairs).values()) if ev_pairs else 0
    eval_patch = _evaluate_full_patch_at_site(
        target_var, ev_pairs, best["layer"], best["component"], best["pos"]
    ) if len(ev_pairs) >= MQ_MIN_EVAL_PAIRS else {
        "iia": float("nan"), "recovery": float("nan"),
        "correct": 0, "n": 0, "p_value": float("nan"),
    }
    eval_majority = _majority_baseline(ev_pairs)
    heldout_patch_ok = (
        eval_patch["n"] > 0 and
        eval_patch["iia"] > max(1/3, eval_majority) and
        eval_patch["p_value"] <= MQ_PATCH_ALPHA
    )
    print(
        f"{target_var} held-out full patch: "
        f"IIA={eval_patch['iia']:.3f} ({eval_patch['correct']}/{eval_patch['n']}), "
        f"p={eval_patch['p_value']:.4f}, recovery={eval_patch['recovery']:.4f}"
    )

    ready_checks = {
        "mq_factual_acc_ok": mq_factual_acc >= MQ_MIN_FACTUAL_ACC,
        "heldout_full_patch_ok": heldout_patch_ok,
        "train_pairs_ok": len(tr_pairs) >= MQ_MIN_TRAIN_PAIRS,
        "eval_pairs_ok": len(ev_pairs) >= MQ_MIN_EVAL_PAIRS,
        "eval_balance_ok": min_eval_per_label >= MQ_MIN_EVAL_PER_LABEL,
    }
    ready = all(ready_checks.values())
    if not ready:
        failed = [k for k, v in ready_checks.items() if not v]
        print(f"WARNING: {target_var} is NOT DAS-ready.")
        print("Failed readiness checks:", failed)
        print("We will not train/report DAS for this variable in the clean notebook.")

    return {
        **best,
        "train_patch_iia": best["patch_iia"],
        "eval_patch_iia": eval_patch["iia"],
        "eval_patch_correct": eval_patch["correct"],
        "eval_patch_n": eval_patch["n"],
        "eval_patch_p_value": eval_patch["p_value"],
        "eval_patch_recovery": eval_patch["recovery"],
        "ready": bool(ready),
        "ready_checks": ready_checks,
        "train_pairs": tr_pairs,
        "eval_pairs": ev_pairs,
        "train_counts": tr_counts,
        "eval_counts": ev_counts,
        "eval_majority": eval_majority,
    }

MQNLI_NEGP = select_mqnli_anchor_and_data("NegP", SEED + 1000)
MQNLI_QPO  = select_mqnli_anchor_and_data("QP_O", SEED + 2000)

MQNLI_READY = {"NegP": MQNLI_NEGP["ready"], "QP_O": MQNLI_QPO["ready"]}

# Keep old variable names for the later cells, but allow None if not ready.
MQ_LAYER_NEGP = MQNLI_NEGP.get("layer")
MQ_COMP_NEGP  = MQNLI_NEGP.get("component")
_negp_pos     = MQNLI_NEGP.get("pos")
_negp_patch_iia = MQNLI_NEGP.get("eval_patch_iia", MQNLI_NEGP.get("patch_iia", float("nan")))
_negp_train_patch_iia = MQNLI_NEGP.get("train_patch_iia", MQNLI_NEGP.get("patch_iia", float("nan")))
negp_tr_pairs = MQNLI_NEGP.get("train_pairs", [])
negp_ev_pairs = MQNLI_NEGP.get("eval_pairs", [])

MQ_LAYER_QPO = MQNLI_QPO.get("layer")
MQ_COMP_QPO  = MQNLI_QPO.get("component")
_qpo_pos     = MQNLI_QPO.get("pos")
_qpo_patch_iia = MQNLI_QPO.get("eval_patch_iia", MQNLI_QPO.get("patch_iia", float("nan")))
_qpo_train_patch_iia = MQNLI_QPO.get("train_patch_iia", MQNLI_QPO.get("patch_iia", float("nan")))
qpo_tr_pairs = MQNLI_QPO.get("train_pairs", [])
qpo_ev_pairs = MQNLI_QPO.get("eval_pairs", [])

print("\nMQNLI DAS readiness:")
print(
    f"  NegP: ready={MQNLI_READY['NegP']}  anchor=(L{MQ_LAYER_NEGP}, {MQ_COMP_NEGP}, pos{_negp_pos})  "
    f"train_anchor_patch_iia={_negp_train_patch_iia:.3f}  heldout_patch_iia={_negp_patch_iia:.3f}"
)
print(
    f"  QP_O: ready={MQNLI_READY['QP_O']}  anchor=(L{MQ_LAYER_QPO}, {MQ_COMP_QPO}, pos{_qpo_pos})  "
    f"train_anchor_patch_iia={_qpo_train_patch_iia:.3f}  heldout_patch_iia={_qpo_patch_iia:.3f}"
)


## 11. DAS on MQNLI internal variables

We only train MQNLI DAS after a stricter prerequisite:

1. A candidate anchor is selected by a balanced, position-aligned full-vector patching sweep on training examples.
2. The selected anchor is re-tested with full-vector patching on held-out, position-aligned, counterfactual-label-balanced eval examples.
3. DAS is skipped unless held-out full-vector patching beats the majority/chance baseline by an exact one-sided binomial test.

This prevents the earlier failure mode where MQNLI DAS was trained and reported even when full activation patching had no causal signal. A low-dimensional DAS intervention cannot be meaningful at a site where replacing the whole residual vector does not work first.


In [ ]:
# ── Section 11b: DAS on MQNLI intermediate variables ────────────────────────
import torch
from torch.utils.data import Dataset
from types import SimpleNamespace

class MQNLICFDataset(Dataset):
    """Counterfactual dataset for MQNLI — compatible with train_das_alignment."""
    def __init__(self, pairs, tokenizer, fixed_pos):
        assert len(pairs) > 0, "Cannot create MQNLICFDataset from an empty pair list."
        self.pairs = pairs
        self.fixed_pos = int(fixed_pos)
        bases  = [p["base"]["prompt"]   for p in pairs]
        srcs   = [p["source"]["prompt"] for p in pairs]
        enc = tokenizer(bases + srcs, padding=True, truncation=True, return_tensors="pt")
        n = len(pairs)
        self._bids   = enc["input_ids"][:n]
        self._sids   = enc["input_ids"][n:]
        self._battn  = enc["attention_mask"][:n]
        self._sattn  = enc["attention_mask"][n:]
        self.examples = [SimpleNamespace(intervention_pos=self.fixed_pos) for _ in pairs]

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        return {
            "base_input_ids":          self._bids[idx],
            "source_input_ids":        self._sids[idx],
            "base_attention_mask":     self._battn[idx],
            "source_attention_mask":   self._sattn[idx],
            "base_label_id":           torch.tensor(self.pairs[idx]["base"]["label_id"], dtype=torch.long),
            "source_label_id":         torch.tensor(self.pairs[idx]["source"]["label_id"], dtype=torch.long),
            "counterfactual_label_id": torch.tensor(self.pairs[idx]["cf_label_id"], dtype=torch.long),
            "intervention_pos":        torch.tensor(self.fixed_pos, dtype=torch.long),
            "target_variable":         self.pairs[idx]["target_var"],
        }

def _cf_label_dist(pairs):
    return dict(sorted(Counter(int(p["cf_label_id"]) for p in pairs).items()))

def _majority_baseline_pairs(pairs):
    if not pairs:
        return float("nan")
    c = Counter(int(p["cf_label_id"]) for p in pairs)
    return max(c.values()) / len(pairs)

def make_random_source_pairs(ev_pairs, target_var, seed):
    """Generalization check: recompute the CF label for a new valid random source."""
    rng = _random.Random(seed)
    out = []
    for p in ev_pairs:
        src = rng.choice(ev_pairs)["source"]
        cf_id = run_mqnli(p["base"], {target_var: src[target_var]})["label_id"]
        out.append(dict(p, source=src, cf_label_id=int(cf_id)))
    return out

def _nan_result():
    return {"iia": float("nan"), "factual_accuracy": float("nan"), "verbalizer_hit_rate": float("nan")}

def train_and_eval_mqnli_var(name, ready, tr_pairs, ev_pairs, layer, component, pos, seed):
    print(f"\n=== MQNLI DAS for {name} ===")
    print(f"ready={ready}, anchor=(L{layer}, {component}, pos{pos})")
    print(f"train={len(tr_pairs)} dist={_cf_label_dist(tr_pairs)}")
    print(f"eval ={len(ev_pairs)} dist={_cf_label_dist(ev_pairs)}")
    print(f"eval majority-label baseline={_majority_baseline_pairs(ev_pairs):.3f}")

    if not ready:
        print(f"Skipping {name}: no methodologically valid full-patching anchor / balanced CF set.")
        return None, _nan_result(), _nan_result()

    tr_ds = MQNLICFDataset(tr_pairs, tokenizer, pos)
    ev_ds = MQNLICFDataset(ev_pairs, tokenizer, pos)

    torch.manual_seed(seed)
    cfg = make_das_config(mq_model, layer=layer, component=component,
                          intervention_dim=16, unit="pos")
    out = train_das_alignment(
        mq_model, tokenizer, tr_ds, cfg,
        num_epochs=20, lr=1e-3, batch_size=8, device=DEVICE,
        verbalizer=verbalizer, fixed_position=pos,
        eval_cf_dataset=ev_ds, progress=True,
    )
    print(f"\n{name} DAS — last 3 epochs:")
    print(pd.DataFrame(out.history).tail(3).to_string(index=False))

    res = evaluate_das_iia(out.intervenable, ev_ds, tokenizer,
                           device=DEVICE, verbalizer=verbalizer,
                           fixed_position=pos)

    rand_pairs = make_random_source_pairs(ev_pairs, name, seed + 123)
    rand_ds = MQNLICFDataset(rand_pairs, tokenizer, pos)
    res_rand = evaluate_das_iia(out.intervenable, rand_ds, tokenizer,
                                device=DEVICE, verbalizer=verbalizer,
                                fixed_position=pos)

    # Warnings that protect against misleading IIA interpretation.
    maj = _majority_baseline_pairs(ev_pairs)
    if not np.isnan(maj) and res["iia"] <= maj + 0.05:
        print(f"WARNING: {name} DAS IIA ({res['iia']:.3f}) does not clearly beat the majority-CF baseline ({maj:.3f}).")
    if res["factual_accuracy"] < 0.80:
        print(f"WARNING: {name} base factual accuracy on DAS examples is {res['factual_accuracy']:.3f}; interpret cautiously.")

    return out, res, res_rand

negp_out, negp_res, negp_res_rand = train_and_eval_mqnli_var(
    "NegP", MQNLI_READY.get("NegP", False),
    negp_tr_pairs, negp_ev_pairs,
    MQ_LAYER_NEGP, MQ_COMP_NEGP, _negp_pos,
    SEED,
)

qpo_out, qpo_res, qpo_res_rand = train_and_eval_mqnli_var(
    "QP_O", MQNLI_READY.get("QP_O", False),
    qpo_tr_pairs, qpo_ev_pairs,
    MQ_LAYER_QPO, MQ_COMP_QPO, _qpo_pos,
    SEED + 1,
)

mq11_summary = pd.DataFrame([
    {"variable": "NegP", "condition": "Trained DAS",      "IIA": negp_res["iia"]},
    {"variable": "NegP", "condition": "Random-source",    "IIA": negp_res_rand["iia"]},
    {"variable": "NegP", "condition": "Majority baseline","IIA": _majority_baseline_pairs(negp_ev_pairs)},
    {"variable": "QP_O", "condition": "Trained DAS",      "IIA": qpo_res["iia"]},
    {"variable": "QP_O", "condition": "Random-source",    "IIA": qpo_res_rand["iia"]},
    {"variable": "QP_O", "condition": "Majority baseline","IIA": _majority_baseline_pairs(qpo_ev_pairs)},
]).round(3)
print("\n", mq11_summary.to_string(index=False))
mq11_summary.to_csv("outputs/tables/mqnli_das_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
labels = ["NegP", "QP_O"]
trained = [negp_res["iia"], qpo_res["iia"]]
randoms = [negp_res_rand["iia"], qpo_res_rand["iia"]]
majority = [_majority_baseline_pairs(negp_ev_pairs), _majority_baseline_pairs(qpo_ev_pairs)]
x = np.arange(len(labels)); w = 0.25
ax.bar(x - w, trained, w, label="Trained DAS")
ax.bar(x, randoms, w, label="Random-source")
ax.bar(x + w, majority, w, label="Majority-CF baseline")
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="balanced chance")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.set_title("MQNLI DAS only reported after held-out full-patching gate")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig("outputs/figures/mqnli_das.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Composition — `do(NegP=src_A, QP_O=src_B)`

The two DAS rotations were trained **independently** (different train sets, different random seeds for the optimizer). The question is whether they **compose**: if we apply both interchange interventions simultaneously — patching NegP from source A *and* QP_O from source B — does the model's output match the symbolic counterfactual label computed by `run_mqnli(base, {NegP: A.NegP, QP_O: B.QP_O})`?

This test is stronger than either single-variable intervention because:
1. The two subspaces must be **disentangled** — overlap would cause interference.
2. The symbolic model predicts the label from two independently changed variables.
3. Any alignment-capacity effect (Makelov/Sutter critique) is unlikely to explain both interventions simultaneously aligning with the correct joint CF label.

pyvene allows multiple simultaneous interventions by specifying two `IntervenableConfig` entries at the same `(layer, position)` site.

In [ ]:
# ── Section 12: Composition — do(NegP=srcA, QP_O=srcB) ──────────────────────
import pyvene as pv

NEG_READY = bool(MQNLI_READY.get("NegP", False)) and negp_out is not None and not np.isnan(negp_res["iia"])
QPO_READY = bool(MQNLI_READY.get("QP_O", False)) and qpo_out is not None and not np.isnan(qpo_res["iia"])

SAME_COMPOSITION_SITE = (
    NEG_READY and QPO_READY and
    MQ_LAYER_NEGP == MQ_LAYER_QPO and
    MQ_COMP_NEGP  == MQ_COMP_QPO and
    int(_negp_pos) == int(_qpo_pos)
)
print("Composition-site check:", {
    "NegP_ready": NEG_READY,
    "QP_O_ready": QPO_READY,
    "NegP": (MQ_LAYER_NEGP, MQ_COMP_NEGP, _negp_pos),
    "QP_O": (MQ_LAYER_QPO,  MQ_COMP_QPO,  _qpo_pos),
    "same_site": SAME_COMPOSITION_SITE,
})

if not (NEG_READY and QPO_READY):
    print("Skipping composition: at least one MQNLI single-variable DAS result is not methodologically valid.")
    joint_iia = float("nan")
    comp_df = pd.DataFrame([
        {"condition": "NegP alone", "IIA": negp_res["iia"]},
        {"condition": "QP_O alone", "IIA": qpo_res["iia"]},
        {"condition": "Composition skipped: invalid single-variable DAS", "IIA": np.nan},
    ])
    comp_df.to_csv("outputs/tables/composition.csv", index=False)

elif not SAME_COMPOSITION_SITE:
    print("Skipping composition: NegP and QP_O selected different DAS sites.")
    print("This is not a failure; it means this simple same-site composition test is not applicable.")
    joint_iia = float("nan")
    comp_df = pd.DataFrame([
        {"condition": "NegP alone", "IIA": negp_res["iia"]},
        {"condition": "QP_O alone", "IIA": qpo_res["iia"]},
        {"condition": "Composition skipped: different sites", "IIA": np.nan},
    ])
    comp_df.to_csv("outputs/tables/composition.csv", index=False)

else:
    MQ_LAYER = MQ_LAYER_NEGP
    MQ_COMP  = MQ_COMP_NEGP
    MQ_DIM   = 16
    MQ_POS   = int(_negp_pos)

    def build_joint_cf(exs, n=256, require_change=True, seed=0):
        """Build triples (base, src_negp, src_qpo) with joint CF label."""
        rng = _random.Random(seed)
        out = []
        seen = set()
        for _ in range(n * 500):
            if len(out) >= n:
                break
            base  = rng.choice(exs)
            src_n = rng.choice(exs)
            src_q = rng.choice(exs)
            if _qpos(base) != MQ_POS or _qpos(src_n) != MQ_POS or _qpos(src_q) != MQ_POS:
                continue
            if base["NegP"] == src_n["NegP"] and base["QP_O"] == src_q["QP_O"]:
                continue
            cf = run_mqnli(base, {"NegP": src_n["NegP"], "QP_O": src_q["QP_O"]})
            if require_change and cf["label_id"] == base["label_id"]:
                continue
            key = (base["prompt"], src_n["prompt"], src_q["prompt"], cf["label_id"])
            if key in seen:
                continue
            seen.add(key)
            out.append({"base": base, "src_negp": src_n, "src_qpo": src_q,
                        "cf_label_id": int(cf["label_id"])})
        return out

    joint_pairs = build_joint_cf(mqnli_eval, n=256, seed=SEED + 200)
    print(f"Joint CF pairs after position filter: {len(joint_pairs)}")
    print("Joint CF-label dist:", dict(Counter(int(p["cf_label_id"]) for p in joint_pairs)))
    assert len(joint_pairs) >= 32, "Too few aligned joint CF pairs for composition."

    _base_prs = [p["base"]["prompt"]     for p in joint_pairs]
    _srcn_prs = [p["src_negp"]["prompt"] for p in joint_pairs]
    _srcq_prs = [p["src_qpo"]["prompt"]  for p in joint_pairs]
    enc_j = tokenizer(_base_prs + _srcn_prs + _srcq_prs,
                      padding=True, truncation=True, return_tensors="pt")
    nj = len(joint_pairs)
    _j_bids  = enc_j["input_ids"][:nj].to(DEVICE)
    _j_nids  = enc_j["input_ids"][nj:2*nj].to(DEVICE)
    _j_qids  = enc_j["input_ids"][2*nj:].to(DEVICE)
    _j_battn = enc_j["attention_mask"][:nj].to(DEVICE)
    _j_nattn = enc_j["attention_mask"][nj:2*nj].to(DEVICE)
    _j_qattn = enc_j["attention_mask"][2*nj:].to(DEVICE)
    _j_cf    = torch.tensor([p["cf_label_id"] for p in joint_pairs], dtype=torch.long, device=DEVICE)

    negp_rotation = list(negp_out.intervenable.interventions.values())[0]
    qpo_rotation  = list(qpo_out.intervenable.interventions.values())[0]
    from nli_das.das import _pick_intervention_class, decode_label
    _cls, _lr = _pick_intervention_class()

    joint_config = pv.IntervenableConfig(
        model_type=type(mq_model),
        representations=[
            pv.RepresentationConfig(MQ_LAYER, MQ_COMP, "pos", 1,
                                    **({} if not _lr else {"low_rank_dimension": MQ_DIM})),
            pv.RepresentationConfig(MQ_LAYER, MQ_COMP, "pos", 1,
                                    **({} if not _lr else {"low_rank_dimension": MQ_DIM})),
        ],
        intervention_types=_cls,
    )
    joint_iv = pv.IntervenableModel(joint_config, mq_model)
    joint_iv.set_device(DEVICE)
    joint_iv.disable_model_gradients()

    _ivs = list(joint_iv.interventions.values())
    assert hasattr(_ivs[0], "rotate_layer") and hasattr(negp_rotation, "rotate_layer"), (
        "Could not copy trained rotation weights; refusing to evaluate random composition."
    )
    _ivs[0].rotate_layer.weight.data.copy_(negp_rotation.rotate_layer.weight.data)
    _ivs[1].rotate_layer.weight.data.copy_(qpo_rotation.rotate_layer.weight.data)
    print("Copied both trained rotation weights into the joint intervention model.")

    joint_correct = 0
    with torch.no_grad():
        for s in range(0, nj, 8):
            e   = min(s + 8, nj)
            bi  = {"input_ids": _j_bids[s:e], "attention_mask": _j_battn[s:e]}
            sni = {"input_ids": _j_nids[s:e], "attention_mask": _j_nattn[s:e]}
            sqi = {"input_ids": _j_qids[s:e], "attention_mask": _j_qattn[s:e]}
            try:
                _, cf_out = joint_iv(bi, [sni, sqi], {"sources->base": (MQ_POS, MQ_POS)})
            except Exception as exc:
                raise RuntimeError(
                    "Simultaneous pyvene composition failed. Not reporting a fake sequential fallback."
                ) from exc
            preds = decode_label(cf_out.logits, verbalizer, attention_mask=_j_battn[s:e])
            joint_correct += int((preds == _j_cf[s:e]).sum().item())

    joint_iia = joint_correct / nj
    print("Simultaneous joint intervention applied successfully.")
    print(f"\n{'Condition':<30s}  IIA")
    print(f"  {'Single DAS (NegP)':<28s}  {negp_res['iia']:.3f}")
    print(f"  {'Single DAS (QP_O)':<28s}  {qpo_res['iia']:.3f}")
    print(f"  {'Composition do(NegP,QP_O)':<28s}  {joint_iia:.3f}")
    print(f"  {'Chance (3-class)':<28s}  {1/3:.3f}")

    comp_df = pd.DataFrame([
        {"condition": "NegP alone",               "IIA": negp_res["iia"]},
        {"condition": "QP_O alone",               "IIA": qpo_res["iia"]},
        {"condition": "Composition (NegP, QP_O)", "IIA": joint_iia},
    ]).round(3)
    comp_df.to_csv("outputs/tables/composition.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(comp_df["condition"], comp_df["IIA"])
    ax.axhline(1/3, color="black", linestyle="--", lw=1, label="chance")
    ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
    ax.set_title("Composition: do(NegP=srcA, QP_O=srcB)")
    ax.legend(); fig.tight_layout()
    fig.savefig("outputs/figures/composition.png", dpi=150, bbox_inches="tight")
    plt.show()

    del joint_iv
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()


## 13. Calibration — random-init GPT-2 control

This repeats the lexical-relation DAS setup on a randomly initialized GPT-2. The point is to check whether DAS can manufacture high IIA even when the model has no learned task structure.

Interpretation:
- random-init IIA near chance means the lexical result depends on learned model structure;
- random-init IIA close to the fine-tuned score would mean the DAS map is too flexible to trust;
- the conservative learned-structure estimate is roughly `fine_tuned_iia - random_init_iia`.


In [ ]:
# ── Section 13: Calibration — random-init GPT-2 control ─────────────────────
# Replicate the §7 DAS setup identically, but use a randomly-initialized GPT-2.
# This bounds the "alignment capacity" effect.

torch.manual_seed(SEED + 999)
random_model = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
# Re-initialize all weights from scratch (zeroing pretrained weights).
random_model.apply(lambda m: m.reset_parameters() if hasattr(m, "reset_parameters") else None)
for p in random_model.parameters():
    p.requires_grad_(False)
random_model.eval()
print("Random-init GPT-2 created (all weights re-initialized).")

# Factual accuracy of the random model (should be ~33%)
_rng_correct = 0
with torch.no_grad():
    for ex, (pids, _) in zip(factual_eval, _encode_label_only(factual_eval, tokenizer, LABEL_STR)):
        logits = random_model(input_ids=torch.tensor([pids]).to(DEVICE)).logits[0, -1]
        pred   = logits[verb_tids].argmax().item()
        _rng_correct += int(pred == ex.label_id)
rng_factual_acc = _rng_correct / len(factual_eval)
print(f"Random-init factual accuracy (expect ~33%): {rng_factual_acc:.1%}")

# Train DAS on random model at the same (BEST_LAYER, COMPONENT, FIXED_POSITION) as §7
torch.manual_seed(SEED)
rng_cfg  = make_das_config(random_model, layer=BEST_LAYER, component=COMPONENT,
                            intervention_dim=DIM, unit="pos")
rng_out  = train_das_alignment(
    random_model, tokenizer, train_ds_f, rng_cfg,
    num_epochs=NUM_EPOCHS, lr=1e-3, batch_size=8,
    device=DEVICE, verbalizer=verbalizer,
    fixed_position=FIXED_POSITION,
    eval_cf_dataset=eval_ds_f,
    progress=True,
)
rng_res  = evaluate_das_iia(rng_out.intervenable, eval_ds_f, tokenizer,
                              device=DEVICE, verbalizer=verbalizer,
                              fixed_position=FIXED_POSITION)

calibration_iia = rng_res["iia"]
trained_iia     = res["iia"]   # from §7

print(f"\nCalibration summary:")
print(f"  Fine-tuned GPT-2 DAS IIA:   {trained_iia:.3f}")
print(f"  Random-init GPT-2 DAS IIA:  {calibration_iia:.3f}")
print(f"  Gap (conservative learned):  {max(0, trained_iia - calibration_iia):.3f}")

calib_df = pd.DataFrame([
    {"model": "Fine-tuned GPT-2",   "IIA": trained_iia},
    {"model": "Random-init GPT-2",  "IIA": calibration_iia},
]).round(3)
calib_df.to_csv("outputs/tables/calibration.csv", index=False)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(calib_df["model"], calib_df["IIA"], color=["tab:blue", "tab:red"])
ax.axhline(1/3, color="black", linestyle="--", lw=1, label="chance (1/3)")
ax.set_ylabel("IIA (lexical_relation)"); ax.set_ylim(0, 1.05)
ax.set_title("Calibration: DAS IIA on fine-tuned vs random-init GPT-2")
ax.legend(); fig.tight_layout()
fig.savefig("outputs/figures/calibration.png", dpi=150, bbox_inches="tight")
plt.show()

del random_model, rng_out
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()
print("\nAll sections complete. Figures saved to outputs/figures/, tables to outputs/tables/.")